Обработайте набор данных, подготовив его к решению задачи классификации/регрессии. Выделите целевой признак, указанный в description.txt вашего варианта. Разделите набор данных на обучающую, тестовую, валидационную выборки.

Постройте различные ансамбли моделей для решения задачи (бэггинг, бустинг деревьев). Оцените их качество (F1 и AUC для классификации / RMSE и R2 для регрессии) на тестовой выборке. Отберите лучшие признаки.

С помощью GridSearch переберите различные комбинации гиперпараметров: определите лучшее количество деревьев n_estimators с точностью до 10. Какой из классификаторов оказывается лучше?

Постройте блендинг(стекинг) нескольких успешных моделей: есть ли улучшение результата?


In [22]:
# Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn import metrics
from sklearn.metrics import roc_auc_score

In [23]:
# Загрузка данных
df=pd.read_csv("data.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [24]:
# Обработка категориальных признаков, имеющих 2 уникальных значения
from sklearn.preprocessing import LabelEncoder

lab_encoder = LabelEncoder()
df["gender"] = lab_encoder.fit_transform(df["gender"])
df["Partner"]=lab_encoder.fit_transform(df["Partner"])
df["PhoneService"] = lab_encoder.fit_transform(df["PhoneService"])
df["Dependents"] = lab_encoder.fit_transform(df["Dependents"])
df["PaperlessBilling"] = lab_encoder.fit_transform(df["PaperlessBilling"])
df["Churn"] = lab_encoder.fit_transform(df["Churn"])

In [25]:
# Обработка категориальных признаков, имеющих более 2 уникальных значений
onehot = OneHotEncoder(drop='first', sparse_output=False)

cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
        'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
        'Contract', 'PaymentMethod']

encoded = onehot.fit_transform(df[cols])
encoded_df = pd.DataFrame(encoded, columns=onehot.get_feature_names_out(cols), index=df.index)
df = pd.concat([df.drop(columns=cols), encoded_df], axis=1)

In [26]:
# Обработка признаков
df = df.drop(columns=['customerID'])
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [27]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,PaymentMethod_nan
0,0,0,1,0,1,0,1,29.85,29.85,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,1,0,0,0,2,1,1,53.85,108.15,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0,0,0,0,2,1,1,70.70,151.65,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [28]:
# Разделение признаков на тренировочную, тествую и валидационную выборки
X = df.drop(['Churn'], axis=1)
y = df['Churn']

from sklearn.model_selection import train_test_split

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    random_state=42,
    stratify=y_train_val
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (4225, 32)
Validation: (1409, 32)
Test: (1409, 32)


In [29]:
# Обработка нулей в признаке TotalCharges (замена на медиану)
# Есть избежание утечки данных
median_train = X_train['TotalCharges'].median()
X_train['TotalCharges'] = X_train['TotalCharges'].fillna(median_train)
X_val['TotalCharges'] = X_val['TotalCharges'].fillna(median_train)
X_train_val['TotalCharges'] = X_train_val['TotalCharges'].fillna(median_train)
X_test['TotalCharges'] = X_test['TotalCharges'].fillna(median_train)

In [30]:
# Обучение с помощью метода случайного леса
clf = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_val)
print('F1 = ', metrics.f1_score(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_pred))

F1 =  0.5220228384991843
ROC-AUC: 0.6757394921077785


In [31]:
# Обучение с помощью метода адаптивного бустинга
from sklearn.ensemble import AdaBoostClassifier

stump = DecisionTreeClassifier(max_depth=1)
ada = AdaBoostClassifier(estimator=stump, n_estimators=3, random_state=42).fit(X_train, y_train)

y_pred = ada.predict(X_val)
print('F1 = ', metrics.f1_score(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_pred))

F1 =  0.5526881720430108
ROC-AUC: 0.6991384432560902


In [32]:
# Обучение с помощью метода XGBoost
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=100, max_depth=8, learning_rate=1)
xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_val)
print('F1 = ', metrics.f1_score(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_pred))

F1 =  0.519774011299435
ROC-AUC: 0.6735255366968922


In [33]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 51.1 MB/s  0:00:01


In [34]:
# Обучение с помощью CatBoost
from catboost import CatBoostClassifier

cat = CatBoostClassifier(iterations=1000, learning_rate=0.1, depth=6, logging_level='Silent')
cat.fit(X_train, y_train)

y_pred = cat.predict(X_val)
print('F1 = ', metrics.f1_score(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_pred))

F1 =  0.5164179104477612
ROC-AUC: 0.6718631326048206


In [35]:
# Кросс-валидация для проверки наличия сильных выбросов
from sklearn.model_selection import cross_val_score

scores = cross_val_score(clf, X, y, cv=5, scoring='f1')
scores

array([0.52805281, 0.52086811, 0.45360825, 0.49403748, 0.50847458])

In [36]:
# Важность признаков в случайном лесу
random_forest = RandomForestClassifier(random_state=1,max_depth=10)
random_forest.fit(X_train, y_train)

importances = pd.DataFrame({'feature':X_train.columns,'importance':np.round(random_forest.feature_importances_,3)})
importances = importances.sort_values('importance',ascending=False)

importances

,feature,importance
4,tenure,0.179
8,TotalCharges,0.172
7,MonthlyCharges,0.118
11,InternetService_Fiber optic,0.070
27,Contract_Two year,0.061
29,PaymentMethod_Electronic check,0.047
26,Contract_One year,0.032
14,OnlineSecurity_Yes,0.031
20,TechSupport_Yes,0.027
6,PaperlessBilling,0.023


In [37]:
# Жадный поиск гиперпараметров
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.1, 0.2],
}

grid = GridSearchCV(
    XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1,
    ),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Лучшие параметры:", grid.best_params_)
print("Лучший CV F1:", grid.best_score_)

best_xgb = grid.best_estimator_

y_val_pred = best_xgb.predict(X_val)

print("Random Forest Validation F1:", metrics.f1_score(y_val, y_val_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_val_pred))

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Лучшие параметры: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}
Лучший CV F1: 0.5786972933626326
Random Forest Validation F1: 0.5560675883256528
ROC-AUC: 0.6956017980314655


In [38]:
# Обучение с помощью метода опорных векторов
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import StackingClassifier

clf = make_pipeline(StandardScaler(), LinearSVC(random_state=42))

clf.fit(X_train, y_train)
y_pred = clf.predict(X_val)
print('SVC: F1 = ', metrics.f1_score(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_pred))

SVC: F1 =  0.5905044510385756
ROC-AUC: 0.7172505102172623


In [41]:
# Обучение с помощью стекинга
estimators = [
    ('xg', XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.1)),
    ('svc', make_pipeline(StandardScaler(), LinearSVC(random_state=42))),
    ('adb', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=3, random_state=42))
]

stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5
)

stacking.fit(X_train, y_train)

y_pred = stacking.predict(X_val)

print("Stacking F1:", metrics.f1_score(y_val, y_pred))

Stacking F1: 0.5852187028657617


In [ ]:
# Оценка обученной модели на тестовой выборке
stacking.fit(X_train_val, y_train_val)
y_pred = stacking.predict(X_test)

print("Stacking F1:", metrics.f1_score(y_test, y_pred))